# Neural Quantum States for Quantum Many-Body Systems: A Variational Study of the Heisenberg Chain

#### This project investigates neural-network representations of quantum many-body wavefunctions using variational Monte Carlo. The one-dimensional Heisenberg/XXZ spin chain will serve as a controlled testbed, with exact diagonalization and matrix-product-state/DMRG calculations providing benchmarks. The project aims to understand both the theoretical foundations and computational implementation of neural quantum states, and to compare their properties with tensor-network representations.

# Neural Quantum States for Quantum Many-Body Systems

## 1. Physical Problem

We consider a one-dimensional isotropic spin-$\frac{1}{2}$ Heisenberg chain. The Hamiltonian is

$$
H = J\sum_{i=1}^{N} \mathbf{S}_i\cdot\mathbf{S}_{i+1},
$$

where

$$
\mathbf{S}_i\cdot\mathbf{S}_{i+1}
=
S_i^xS_{i+1}^x+
S_i^yS_{i+1}^y+
S_i^zS_{i+1}^z.
$$

For the main part of this project we take $J=1$ and $\hbar = 1$ and consider the antiferromagnetic case.

The goal is to study how the ground state of this interacting quantum many-body system can be represented and approximated using different computational approaches, with particular emphasis on neural quantum states and variational Monte Carlo.

We will use exact diagonalization (ED) and matrix-product-state/density-matrix-renormalization-group (MPS/DMRG) calculations as benchmarks.

The central questions of the project are:

1. How can a many-body wavefunction be represented by a neural network?
2. How can its variational parameters be optimized without explicitly constructing the exponentially large Hilbert-space vector?
3. How accurately can a neural quantum state approximate the ground state?
4. How does its performance compare with MPS/DMRG?
5. What can these different representations teach us about the structure of quantum many-body states?


## 2. Exact Diagonalization

### 2.1 Derivation of the Hamiltonian's action on the computational basis 
A prominent feature of the system we are considering is that, since the Hamiltonian contains only nearest-neighbor interactions, each basis state is coupled by $H$ only to itself and to basis states obtained by exchanging neighboring opposite spins. Since there are only $N-1$ nearest-neighbor pairs, each column of the Hamiltonian contains at most $N$ nonzero entries. Consequently, when represented in matrix form, the Hamiltonian is sparse, which greatly simplifies diagonalization and reduces the computational effort significantly. Let us begin by constructing the Hamiltonian. We will attempt to find the matrix form of the Hamiltonian by finding its action on the basis states of the Hilbert space, i.e., our construction will build the Hamiltonian column by column.

Suppose we have a spin chain composed of $N$ spin-$\frac{1}{2}$ particles (electrons, for instance). The spin states are denoted by $\left| \uparrow \right\rangle$ and $\left| \downarrow \right\rangle$ for up spins and down spins, respectively. To represent the states computationally, we will use 1 for $\left|\uparrow\right\rangle$ and 0 for $\left|\downarrow\right\rangle$. So, for instance,

$$
\left| \uparrow \uparrow \uparrow \downarrow \downarrow \right\rangle
\equiv
\left| 11100 \right\rangle
$$

will be understood as spins 1, 2 and 3 up and spins 4 and 5 down.

The Hilbert space of the composite system is simply the tensor product of the single-particle Hilbert spaces on the chain. An arbitrary basis state will be denoted by

$$
\left| \sigma \right\rangle
=
\left| \sigma_{1} \sigma_{2} \cdots \sigma_{N} \right\rangle
$$

where each $\sigma_{i} \in \{1,0\}$.

The spin operators for each particle are constant multiples of the Pauli matrices:

$$
\left( S_{x}, S_{y}, S_{z} \right)
=
\dfrac{\hbar}{2}
\left( \sigma_{x}, \sigma_{y}, \sigma_{z} \right)
$$

We will assume $\hbar = 1$ as stated before. Taking $J=1$ as stated before, the Hamiltonian is given by

$$
H =
\sum_{i=1}^{N-1}
\mathbf{S}_{i} \cdot \mathbf{S}_{i+1}
=
\sum_{i=1}^{N-1} h_{i}
$$

where

$$
h_{i} := \mathbf{S}_{i} \cdot \mathbf{S}_{i+1}
$$

which may be called the $\emph{local interaction term}$.

$$
h_{i}
=
\mathbf{S}_{i} \cdot \mathbf{S}_{i+1}
=
S_i^x S_{i+1}^x
+
S_i^y S_{i+1}^y
+
S_i^z S_{i+1}^z
$$

Working with the raising and lowering operators $S^{\pm}$ is more straightforward than $S_x$ and $S_y$ because their action on the computational basis states is known. We have

$$
S^{\pm} = S^{x} \pm i S^{y}
$$

From which, we get

$$
S^{x}_{j}
=
\dfrac{1}{2}
\left(
S^{+}_{j} + S^{-}_{j}
\right)
\qquad , \qquad
S^{y}_{j}
=
\dfrac{1}{2i}
\left(
S^{+}_{j} - S^{-}_{j}
\right)
$$

Substitution and simplification will give

$$
h_{i}
=
\dfrac{1}{2}
\left(
S^{+}_{i} S^{-}_{i+1}
+
S^{-}_{i} S^{+}_{i+1}
\right)
+
S^{z}_{i} S^{z}_{i+1}
$$

The action of the raising and lowering operators is as follows:

\begin{align}\notag
&S^{+} \left| 0 \right\rangle = \left| 1 \right\rangle
\qquad &&, \qquad
S^{+} \left| 1 \right\rangle = 0
\\ \notag
&S^{-} \left| 0 \right\rangle = 0
\qquad &&, \qquad
S^{-} \left| 1 \right\rangle = \left| 0 \right\rangle
\end{align}

How does $h_i$ act on an arbitrary basis state?

$$
h_{i}
\left| \cdots \sigma_{i} \sigma_{i+1} \cdots \right\rangle
=
\left(
\dfrac{1}{2}
\left(
S^{+}_{i} S^{-}_{i+1}
+
S^{-}_{i} S^{+}_{i+1}
\right)
+
S^{z}_{i} S^{z}_{i+1}
\right)
\left| \cdots \sigma_{i} \sigma_{i+1} \cdots \right\rangle
$$

If the spins $i$ and $i+1$ are aligned, i.e., they are both either up or down, we get

$$
h_{i}
\left| \cdots \sigma_{i} \sigma_{i+1} \cdots \right\rangle
=
\dfrac{1}{4}
\left| \cdots \sigma_{i} \sigma_{i+1} \cdots \right\rangle
$$

And if they have opposite orientations, we get

$$
h_{i}
\left| \cdots \sigma_{i} \sigma_{i+1} \cdots \right\rangle
=
-\dfrac{1}{4}
\left| \cdots \sigma_{i} \sigma_{i+1} \cdots \right\rangle
+
\dfrac{1}{2}
\left| \cdots \sigma_{i+1} \sigma_{i} \cdots \right\rangle
$$

In the second ket, the spins for sites $i$ and $i+1$ get switched.

Let us consider a simple case of $N=4$ for the sake of illustration and suppose that we want to find the action of $H$ on the given basis state $\left|1010\right\rangle$.

\begin{align} \notag
H\left|1010\right\rangle
&= \sum_{i=1}^{3}h_i\left|1010\right\rangle \\ \notag
&= h_1\left|1010\right\rangle
+ h_2\left|1010\right\rangle
+ h_3\left|1010\right\rangle \\ \notag
&= -\dfrac{1}{4}\left|1010\right\rangle
+ \dfrac{1}{2}\left|0110\right\rangle \\ \notag
&\quad -\dfrac{1}{4}\left|1010\right\rangle
+ \dfrac{1}{2}\left|1100\right\rangle \\ \notag
&\quad -\dfrac{1}{4}\left|1010\right\rangle
+ \dfrac{1}{2}\left|1001\right\rangle
\end{align}

And hence

$$
H\left|1010\right\rangle
=
-\dfrac{3}{4}\left|1010\right\rangle
+
\dfrac{1}{2}
\left(
\left|0110\right\rangle
+
\left|1100\right\rangle
+
\left|1001\right\rangle
\right)
$$

By finding the action of $H$ on this basis state, we can determine the nonzero elements of the column of $H$ corresponding to $\left|1010\right\rangle$.

Now our objective is to find the matrix representation of a Hamiltonian for a chain of $N$ spins. We will attack the problem by evaluating the effect of $H$ on an arbitrary basis state $\left|\sigma\right\rangle$. The implementation below therefore takes a basis state $\left|\sigma\right\rangle$ as input and returns the nonzero terms and their coefficients in the resulting linear combination $H\left|\sigma\right\rangle$.

Reproducing this process for every basis state gives the complete matrix representation of $H$.

In [166]:
# This cell is to implement the action of the Heisenberg Hamiltonian on an arbitrary basis state |\sigma>.
# The Hamiltonian is given by:
# H = \sum_{i = 1}^{N-1} h_i
# Where
# h_i = S_i \cdot S_{i+1}.
# The action of h_i on arbitrary states has been computed by hand and the result will be implemented here.


import numpy as np

# Some default inputs for N and |\sigma>.
sigma = '1010'

# The basis state should be given as a binary string; '1001...' is to be
# interpreted as up, down, down, up, ....
# We will convert it to a tuple for further manipulation.
def convert_to_tuple(sigma):
    return tuple(int(x) for x in sigma)

# We will store the basis states as decimal integers.
# This function converts a binary basis state to decimal:
def binary_to_decimal(number):
    # The input should be a tuple
    N = len(number)
    powers = [2**(N-i) for i in range(1, N+1)]
    decimal = np.array(number) * np.array(powers)
    return np.sum(decimal)

# H |\sigma> = \sum_{i} h_i |\sigma>.
def apply_H(sigma, n_spins):
    # Store the basis states and their coefficients in a dictionary
    # representing the resulting linear combination.
    lc = {}

    # Convert the basis-state string to a tuple.
    sigma = convert_to_tuple(sigma)

    # Iterate over the nearest-neighbor pairs.
    for i in range(1, n_spins):

        # Depending on sigma[i-1] and sigma[i], the action of h_i is different.
        # If the basis already exists in lc, just add the value
        # else, create the key entry and store the value as well.
        if sigma[i-1] == sigma[i]:
            basis = int(binary_to_decimal(sigma))
            coefficient = 0.25

            if basis in lc:
                lc[basis] += coefficient
            else:
                lc[basis] = coefficient

        else:
            basis_1 = int(binary_to_decimal(sigma))
            coefficient_1 = -0.25

            if basis_1 in lc:
                lc[basis_1] += coefficient_1
            else:
                lc[basis_1] = coefficient_1

            basis_2 = list(sigma)
            basis_2[i-1], basis_2[i] = basis_2[i], basis_2[i-1]
            basis_2 = tuple(basis_2)
            basis_2 = int(binary_to_decimal(basis_2))
            coefficient_2 = 0.5

            if basis_2 in lc:
                lc[basis_2] += coefficient_2
            else:
                lc[basis_2] = coefficient_2

    return lc

In [167]:
# A quick check
print(apply_H(sigma, 4))

{10: -0.75, 6: 0.5, 12: 0.5, 9: 0.5}


### 2.2 Construction and exact diagonalization for a chain of 4 spins

### Constructing the Hamiltonian Matrix

Now that we have determined the action of the Hamiltonian on an arbitrary basis state, we can construct its complete matrix representation. We will first do this using a dense matrix for a small system with $N=4$ spins. This allows us to inspect the matrix directly and verify that our implementation agrees with the results obtained analytically.

The rows and columns of the matrix are indexed by the integers

$$
0,1,\ldots,2^N-1,
$$

which correspond to the computational-basis states written in their binary representation. We construct the matrix column by column; for each column index, we convert the corresponding integer into an $N$-bit binary string, use `apply_H` to evaluate the action of $H$ on that basis state, and then use the resulting dictionary to fill the nonzero entries of the column.

Once the matrix has been constructed, we can diagonalize it numerically to obtain the energy spectrum and the corresponding eigenstates. In particular, the lowest eigenvalue gives the ground-state energy, while its eigenvector gives the ground-state wavefunction in the computational basis.

After verifying the construction for this small system, we will examine why storing the Hamiltonian as a dense matrix quickly becomes impractical as $N$ increases, despite the Hamiltonian being highly sparse. This will motivate the use of sparse matrix representations.

In [168]:
# The rows and columns of H are indexed by the integers
# 0, 1, 2, ..., 2^N - 1. We iterate over the columns and
# apply H to each corresponding basis state. Since apply_H
# takes a binary-string representation of the state, we first
# convert the integer index to an N-bit binary string.

# Specify N:
N = 4

# Initialize H with entries all equal to zero:
H = np.zeros((2**N, 2**N))

# The outer loop iterates over columns, while the inner loop
# fills the nonzero entries of each column.
for i in range(2**N):
    # First, we need to convert the matrix indices from decimals to binaries 
    # because our apply_H function gets entries of that form.
    state = format(i, f'0{N}b')

    # Now that we have the state, we can act apply_H on it and get the
    # desired dictionary to fill the corresponding column.
    lc = apply_H(state, N)

    for key in lc.keys():
        H[key, i] = lc[key]

# Let's see how many nonzero elements H has
print('Number of nonzero elements in H :', np.count_nonzero(H))
print('Total numbe rof elements in H :', 4**N)

# Some quick checks
print(np.allclose(H, H.T))
print(H[:, 10])

# Now let us diagonalize this matrix directly using the numpy linalg module
eigenvalues, eigenvectors = np.linalg.eigh(H)
print(eigenvalues)
E_0 = eigenvalues[0]
psi_0 = eigenvectors[:, 0]
print(np.allclose(H @ psi_0, eigenvalues[0] * psi_0))
print("Ground-state energy:\n", E_0)
print("Ground-state vector:\n", psi_0)

Number of nonzero elements in H : 40
Total numbe rof elements in H : 256
True
[ 0.    0.    0.    0.    0.    0.    0.5   0.    0.    0.5  -0.75  0.
  0.5   0.    0.    0.  ]
[-1.6160254  -0.95710678 -0.95710678 -0.95710678 -0.25       -0.25
 -0.25        0.1160254   0.45710678  0.45710678  0.45710678  0.75
  0.75        0.75        0.75        0.75      ]
True
Ground-state energy:
 -1.6160254037844386
Ground-state vector:
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.49429245e-01
  0.00000000e+00 -5.57677536e-01  4.08248290e-01  0.00000000e+00
  0.00000000e+00  4.08248290e-01 -5.57677536e-01  1.29787663e-18
  1.49429245e-01 -5.26713922e-19  1.41132570e-19  0.00000000e+00]


The construction above gives the complete Hamiltonian matrix for the $N=4$ chain. We can then diagonalize the matrix directly to obtain the full energy spectrum and the corresponding eigenstates. In particular, the lowest eigenvalue gives the ground-state energy, while its corresponding eigenvector gives the ground-state wavefunction in the computational basis.

For $N=4$, the Hilbert space has dimension

$$
d=2^N=16,
$$

so the Hamiltonian is a $16\times16$ matrix. This is small enough that storing and diagonalizing the dense matrix is straightforward. However, the Hilbert-space dimension grows exponentially with the number of spins, while the Hamiltonian itself remains highly sparse. A dense representation therefore stores a very large number of entries that are identically zero.

For a system of $N$ spins, a dense representation requires

$$
(2^N)^2 = 4^N
$$

matrix elements. In contrast, each column of the Heisenberg Hamiltonian contains at most $N$ nonzero elements. It is therefore natural to exploit this sparsity when studying larger systems.

We will next construct the same Hamiltonian using a sparse matrix representation. This will allow us to work with larger chains and will also prepare us for using sparse eigensolvers to obtain the ground state without computing the entire spectrum.

In [169]:
# We will now take a leap towards a more optimized and efficient way of
# storing sparse Hamiltonians. We will use the COO construction from 
# scipy.sparse module.

# Define a function that gets the required list for a COO construction
def coo_lists(apply_H, N):
    """
    :Parameters:
    apply_H : the function that computes the action of H on an arbitrary basis state
    N : number of spins on the chain

    :Returns:
    rows, columns, values for the COO construction
    """
    # Initialize rows, columns and values
    rows, columns, values = [], [], []

    # We will iterate on columns and store the required entries for our lists
    for i in range(2**N):
        # Convert the column index to binary first
        state = format(i, f'0{N}b')

        # Get the dictionary corresponding to this column
        lc = apply_H(state, N)

        # Now store the entries
        for key, value in lc.items():
            rows.append(key)
            columns.append(i)
            values.append(value)

    return rows, columns, values

In [170]:
# A quick check
rows, columns, values = coo_lists(apply_H, N)
print(len(rows))
print(len(columns))
print(len(values))

40
40
40


In [171]:
# Now moving on to the sparse construction for a chain of size N = 4
import scipy as sp

# Create the sparse Hamiltonian
rows, columns, values = coo_lists(apply_H, N)
H_coo = sp.sparse.coo_matrix((values, (rows, columns)), shape = (2**N, 2**N))

# Quick checks
print(H_coo.nnz)
print(np.allclose(H_coo.toarray(), H))

# COO is excellent for constructing the matrix but for operations such as
# multplication or H∣ψ⟩, CSR is a more suitable construction.
# So we will create a CSR version of our Hamiltonian.
H_csr = H_coo.tocsr()

# Verification
print(H_csr.nnz)
print(np.allclose(H_csr.toarray(), H))

40
True
40
True


### 2.3 Using `eigsh` for sparse matrices

In the previous section, we constructed the Hamiltonian as a sparse matrix and converted it to the CSR format. We can now take advantage of this representation when diagonalizing the Hamiltonian. Rather than computing the entire spectrum, as we did with `numpy.linalg.eigh`, we are primarily interested in the ground state, which corresponds to the lowest eigenvalue. SciPy's `eigsh` function allows us to compute only a selected number of eigenpairs of a real symmetric or complex Hermitian sparse matrix, making it particularly useful for this purpose.

In [172]:
# Find the ground-state eigenvalue and eigenvector using a sparse eigensolver
eigenvalues, eigenvectors = sp.sparse.linalg.eigsh(H_csr, k=1, which='SA')
print(eigenvalues)
print(eigenvectors)
print(np.allclose(eigenvalues[0], E_0))
psi_0_sparse = eigenvectors[:, 0]
print(np.allclose(H_csr @ psi_0_sparse,
                  eigenvalues[0] * psi_0_sparse))

[-1.6160254]
[[ 1.11022302e-16]
 [-5.55111512e-17]
 [ 4.16333634e-16]
 [-1.49429245e-01]
 [-1.66533454e-16]
 [ 5.57677536e-01]
 [-4.08248290e-01]
 [ 2.77555756e-17]
 [ 2.77555756e-17]
 [-4.08248290e-01]
 [ 5.57677536e-01]
 [ 5.55111512e-16]
 [-1.49429245e-01]
 [-1.94289029e-16]
 [ 3.33066907e-16]
 [ 6.93889390e-18]]
True
True


### 2.4 Summary and limitations of exact diagonalization

We have constructed the Hamiltonian of the Heisenberg chain directly from its action on the computational basis. For a chain of $N$ spins, the Hilbert-space dimension is $2^{N}$, and although the Hamiltonian is sparse due to the local nature of the interactions, the dimension of the Hilbert space still grows exponentially with system size.

For small systems, exact diagonalization provides direct access to the energy spectrum and eigenstates and therefore serves as an important benchmark for approximate methods. In particular, we can use the exact ground-state energy and wavefunction to assess the accuracy of methods such as DMRG and neural quantum states.

However, the exponential growth of the Hilbert space makes exact diagonalization impractical beyond relatively small system sizes. This motivates the use of methods that can represent or approximate many-body wavefunctions without explicitly storing the full $2^{N}$-dimensional state vector.

## 3. Variational Monte Carlo

The exact diagonalization approach used in the previous section provides an exact solution for sufficiently small systems, but its computational cost grows exponentially with system size. We therefore need alternative approaches that can approximate the ground state without explicitly representing the full many-body wavefunction.

Variational Monte Carlo (VMC) provides one such approach. Rather than solving the eigenvalue problem directly, we introduce a parameterized trial wavefunction and optimize its parameters to minimize the expectation value of the energy.

The basic idea is:

$$
\text{choose a trial wavefunction}
\;\longrightarrow\;
\text{estimate its energy}
\;\longrightarrow\;
\text{optimize its parameters}.
$$

The first step is to understand the variational principle on which this approach is based.

### 3.1 The variational principle

Suppose that the Hamiltonian has eigenstates

$$
H|n\rangle = E_n|n\rangle,
$$

with the ground-state energy satisfying

$$
E_0 \leq E_1 \leq E_2 \leq \cdots.
$$

Consider an arbitrary normalized trial state $|\psi\rangle$. We can expand it in the complete set of energy eigenstates:

$$
|\psi\rangle = \sum_n c_n |n\rangle,
$$

where normalization requires

$$
\sum_n |c_n|^2 = 1.
$$

The expectation value of the energy is

$$
E[\psi]
=
\langle\psi|H|\psi\rangle.
$$

Substituting the expansion of $|\psi\rangle$ gives

$$
E[\psi]
=
\sum_n |c_n|^2 E_n.
$$

Since every eigenenergy satisfies $E_n \geq E_0$,

$$
E[\psi]
\geq
E_0\sum_n |c_n|^2
=
E_0.
$$

Therefore,

$$
\boxed{E[\psi]\geq E_0.}
$$

This is the variational principle: the expectation value of the energy of any normalized trial state provides an upper bound to the exact ground-state energy.

### 3.2 Parameterized trial wavefunctions

In practice, we consider a family of trial wavefunctions parameterized by a set of parameters $\theta$:

$$
|\psi_\theta\rangle.
$$

The corresponding variational energy is

$$
E(\theta)
=
\frac{
\langle\psi_\theta|H|\psi_\theta\rangle
}{
\langle\psi_\theta|\psi_\theta\rangle
}.
$$

The variational principle guarantees that

$$
E(\theta)\geq E_0
$$

for every choice of $\theta$. Therefore, our goal is to find the parameters that minimize the variational energy:

$$
\boxed{
\min_\theta E(\theta).
}
$$

The quality of the resulting approximation depends on the expressive power of the chosen family of trial wavefunctions. In later sections, we will consider two different types of variational representations: matrix product states and neural quantum states.

### 3.3 Monte Carlo estimation of the variational energy

The variational energy can be written in terms of the coefficients of the trial wavefunction as

$$
E(\theta)
=
\dfrac{
\sum_{\sigma,\sigma'}
\psi_\theta^*(\sigma')
H_{\sigma',\sigma}
\psi_\theta(\sigma)
}{
\sum_\sigma |\psi_\theta(\sigma)|^2
}.
$$

The double sum in the numerator and the sum in the denominator both become exponentially expensive as the size of the Hilbert space increases. We therefore want to rewrite the expression in a form that can be estimated by sampling basis states.

We define the local energy of a basis state $\sigma$ as

$$
E_{\mathrm{loc}}(\sigma)
=
\sum_{\sigma'}
H_{\sigma',\sigma}
\frac{\psi_\theta(\sigma')}{\psi_\theta(\sigma)},
$$

where we initially consider real-valued trial wavefunctions.

Using this definition, the variational energy becomes

$$
E(\theta)
=
\frac{
\sum_\sigma |\psi_\theta(\sigma)|^2 E_{\mathrm{loc}}(\sigma)
}{
\sum_\sigma |\psi_\theta(\sigma)|^2
}.
$$

This suggests defining

$$
p_\theta(\sigma)
=
\frac{|\psi_\theta(\sigma)|^2}
{\sum_{\sigma'}|\psi_\theta(\sigma')|^2},
$$

which can be interpreted as a probability distribution over the computational-basis states. The energy can then be written as

$$
E(\theta)
=
\sum_\sigma p_\theta(\sigma)E_{\mathrm{loc}}(\sigma).
$$

Therefore, instead of explicitly summing over all basis states, we can estimate the energy by drawing samples from $p_\theta(\sigma)$ and averaging their local energies.

Before implementing the sampling procedure, we first consider a simple trial wavefunction for which the expectation value can be evaluated directly.

In [173]:
E_avg = []
configs = ['0011', '0101', '0110', '1001', '1010', '1100']
for config in configs:
    lc = apply_H(config, n_spins = 4)
    H_elements = [lc[key] for key in lc]
    E_loc = sum(H_elements)
    E_avg.append(E_loc)

E = (1 / len(E_avg)) * sum(E_avg)

print(E)

0.75


In this test, we use a uniform trial wavefunction over the six basis states in the $S_z=0$ sector. Since all amplitudes are equal, the ratio

$$
\frac{\psi(\sigma')}{\psi(\sigma)}
$$

is equal to one. Therefore, the local energy reduces to the sum of the nonzero Hamiltonian matrix elements generated by `apply_H`.

The resulting energy is therefore the direct average of the local energies over the six configurations. This gives a simple test of the connection between the Hamiltonian action implemented in the exact diagonalization section and the local-energy expression used in VMC.

### 3.4 Constructing a Monte Carlo move

The expression

$$
E(\theta)
=
\sum_\sigma p_\theta(\sigma)E_{\mathrm{loc}}(\sigma)
$$

is useful only if we can obtain configurations distributed according to $p_\theta(\sigma)$.

We therefore need a procedure that moves through the space of computational-basis states and eventually generates samples from this distribution.

For the Heisenberg chain, we choose to work within a fixed $S_z$ sector. The Monte Carlo move should therefore preserve the total $S_z$. A simple way to achieve this is to choose one up spin and one down spin and exchange their positions.

For a state such as

$$
|\sigma\rangle = |1001011101\rangle,
$$

the move selects one position containing $1$ and one position containing $0$ and exchanges them. The number of up and down spins is unchanged, and therefore the total $S_z$ is preserved.

We can now turn this procedure into a function so that it can be used repeatedly during the Monte Carlo sampling.

In [174]:
# We will now work out a function that takes a basis state
# as input and returns another basis state in the same S_{z}
# sector (since total S_{z} must be conserved for this Hamiltonian).

def propose_move(sigma):
    # Choose two random spins (one up and the other one, down)
    # and then exchange them. This way, the total S_{z} is conserved.
    # We will give the input state as a string and store the indices
    # of 1's and 0's in two separate lists and then, we select a random
    # index from each list and swap their corresponding spins.

    proposal = sigma

    # Iterate over the string and store the indices on 1's and 0's
    ones = [i for i, char in enumerate(proposal) if char == '1']
    zeros = [i for i, char in enumerate(proposal) if char == '0']
    # print(ones, zeros)

    # Select one random index from each list
    i, j = np.random.choice(ones), np.random.choice(zeros)

    # Swap the selected spins
    proposal = list(proposal)
    proposal[i], proposal[j] = proposal[j], proposal[i]
    proposal = ''.join(proposal)

    return proposal

### 3.5 Checking the symmetry constraint

Since the proposed move exchanges one up spin with one down spin, it should preserve the total $S_z$ of the state.

For a spin-$1/2$ basis state,

$$
S_z^{\mathrm{tot}}
=
\frac{1}{2}(N_\uparrow-N_\downarrow).
$$

Since

$$
N_\downarrow=N-N_\uparrow,
$$

this can also be written as

$$
S_z^{\mathrm{tot}}
=
N_\uparrow-\frac{N}{2}.
$$

We can use this expression to check that repeated Monte Carlo moves remain within the same $S_z$ sector.

In [175]:
# This function will compute the total S_{z} for an
# arbitrary basis state.

def S_z(sigma):
    # The basis state is passed into the function as
    # a string of binary number.

    # The total S_{z} is equal to (1/2)*(N_up - N_down)
    # S_{z} = (1/2)*(N_up - (N - N_up)) = N_up - N/2.
    # Get the number of up spins
    N_up = sigma.count('1')
    S_z = N_up - 0.5*len(sigma)

    return S_z

In [176]:
# Let's do a sanity to make sure our proposed_move function
# does not take states outside of its corresponding sector.

# Some arbitrary input basis state
sigma = '111001101111'

# Number of samples
n_samples = 10

# Create a list to store the proposals
sigmas = [sigma]
for i in range(n_samples):
    sigma = propose_move(sigma)
    sigmas.append(sigma)

# Check if they all have the same S_{z}
S_z_list = [S_z(state) for state in sigmas]

print(S_z_list)

[3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0]


### 3.6 Metropolis-Hastings sampling

We now need a way to generate configurations distributed according to

$$
p_\theta(\sigma)
=
\frac{|\psi_\theta(\sigma)|^2}
{\sum_{\sigma'}|\psi_\theta(\sigma')|^2}.
$$

For a small system, we could calculate this distribution explicitly by evaluating the wavefunction for every basis state. However, this is not feasible for large systems because the number of basis states grows exponentially with $N$.

The idea behind Monte Carlo sampling is to construct a Markov chain of basis states

$$
\sigma_0 \rightarrow \sigma_1 \rightarrow \sigma_2
\rightarrow \cdots
$$

whose stationary distribution is $p_\theta(\sigma)$. If the chain has reached this stationary distribution, the configurations visited by the chain can be used as samples from $p_\theta$.

To construct such a chain, suppose that the current configuration is $\sigma$. We first propose a new configuration $\sigma'$ according to some proposal distribution

$$
q(\sigma'|\sigma).
$$

The proposed configuration is not always accepted. Instead, we accept it with a probability chosen such that the resulting Markov chain has $p_\theta(\sigma)$ as its stationary distribution. This is the role of the Metropolis-Hastings acceptance rule.

The general acceptance probability is

$$
A(\sigma\rightarrow\sigma')
=
\min\left[
1,
\frac{
p_\theta(\sigma')q(\sigma|\sigma')
}{
p_\theta(\sigma)q(\sigma'|\sigma)
}
\right].
$$

In our case, the proposal consists of exchanging one up spin with one down spin. This proposal is symmetric: the probability of proposing $\sigma'$ from $\sigma$ is the same as the probability of proposing $\sigma$ from $\sigma'$,

$$
q(\sigma'|\sigma)=q(\sigma|\sigma').
$$

Therefore, the proposal probabilities cancel:

$$
A(\sigma\rightarrow\sigma')
=
\min\left[
1,
\frac{p_\theta(\sigma')}{p_\theta(\sigma)}
\right].
$$

The target distribution itself contains an unknown normalization factor. However, this factor is the same for both configurations, so it also cancels in the ratio:

$$
\frac{p_\theta(\sigma')}{p_\theta(\sigma)}
=
\frac{|\psi_\theta(\sigma')|^2}
{|\psi_\theta(\sigma)|^2}.
$$

Thus, the acceptance probability becomes

$$
A(\sigma\rightarrow\sigma')
=
\min\left[
1,
\frac{|\psi_\theta(\sigma')|^2}
{|\psi_\theta(\sigma)|^2}
\right].
$$

This means that we never need to calculate the normalization of the wavefunction. We only need to evaluate its relative weights for the current and proposed configurations.

In [177]:
# A function that implements bra multiplication to find the
# coefficient \psi(\sigma) of the basis state |\sigma> in the
# linear combination of the trial wavefunction |\psi> given by
# \sum_{\sigma} \psi(\sigma) |\sigma>.
# The trial wavefunction is stored as a dictionary that contains
# basis states as keys and their coefficients as their corresponding values.

def bra(trial_psi, basis):
    # trial_psi is the given trial wavefunction and
    # basis (given as a binary string), is the basis
    # whose coefficient we want to get. The idea is to
    # go over the the keys of the dictionary and 
    # capture the value for the key that is equal
    # to our basis state.

    # psi is the coefficient corresponding to basis
    # which we initialize as 0 and change it if the
    # basis is indeed present in the linear combination.
    psi = 0

    if basis in trial_psi:
        psi = trial_psi[basis]

    return psi

In [178]:
# Quick check
trial_psi = {
    '0011': 0.10,
    '0101': 0.30,
    '0110': -0.20,
    '1001': 0.40,
    '1010': 0.50,
    '1100': -0.60
}
print(bra(trial_psi, '1010'))
print(bra(trial_psi, '1100'))
print(bra(trial_psi, '1111'))

0.5
-0.6
0


For a given pair of configurations, we can now calculate the acceptance probability using the ratio of their wavefunction amplitudes.

In [179]:
# In this cell, a function acc_prob is defined that calculates
# the acceptance probability for the Metropolis_hastings algorithm.
# The acceptance probability is given by
# A (\sigma \rightarrow \sigma') = min[1, R]
# where R = \dfrac{|\psi(\sigma')|^2}{|\psi(\sigma)|^2}.

def acc_prob(trial_psi, sigma, sigma_prime):
    # The basis states are given as binary strings
    # and the ratio R is computed and finally, the
    # acceptance probability is computed and returned as
    # output of the function. The trial wavefunction
    # is given as a dictionary as before.

    # Get the coefficients
    psi_sigma, psi_sigma_prime = bra(trial_psi, sigma), bra(trial_psi, sigma_prime)

    # Compute R
    R = (psi_sigma_prime / psi_sigma) ** 2

    # Compute A
    A = min(1, R)

    return A

In [180]:
# Test
trial_psi = {
    '0011': 0.10,
    '0101': 0.30,
    '0110': -0.20,
    '1001': 0.40,
    '1010': 0.50,
    '1100': -0.60
}
sigma        = '1010'
sigma_prime  = '0101'

print(acc_prob(trial_psi, sigma, sigma_prime))
print(acc_prob(trial_psi, sigma_prime, sigma))

0.36
1


### 3.7 Implementing the Metropolis sampler

We can now combine the proposal move and the acceptance rule into a complete Metropolis-Hastings sampler.

At each step, the algorithm:

1. starts from the current configuration $\sigma$;
2. proposes a new configuration $\sigma'$;
3. calculates the acceptance probability $A(\sigma\rightarrow\sigma')$;
4. accepts the proposed configuration with probability $A$;
5. otherwise keeps the current configuration.

The resulting sequence of configurations forms a Markov chain. After an initial equilibration period, the chain can be used to estimate expectation values with respect to $p_\theta(\sigma)$.

We keep the initial part of the chain separately as burn-in samples and use the remaining configurations as the production samples.

In [181]:
# The metropolis_sampler is defined to sample from
# a distribution p(\sigma) when its normalization factor is
# not known or it's difficult to evaluate. The trial
# wavefunction is passed to the function and some random
# initial \sigma is given as the starting point and then,
# moves are proposed and they are either accepted or rejected.
# In the meantime, the samples are stored and after excluding
# the burn-ins, we hope to have obtained a Markov chain that
# has the stationary distribution p(\sigma).

import numpy as np
np.random.seed(24)

def metropolis_sampler(psi, N, n, initial_state = 'random'):
    """
    :Parameters:
    psi: the trial wavefunction (dictionary)
    N: number of Metropolis iteration (integer)
    n: number of burn-in samples to discard (integer)

    :returns:
    burn_ins: the list of burn_ins (list)
    samples: the list of samples obtained using the Metropolis-Hastings algorithm (list)
    acceptance_rate: The portion of accepted samples
    """

    # Initialize the list to store the samples
    samples = []

    # Select some basis state at random if initial_state = 'random'
    if initial_state == 'random':
        sigma = np.random.choice(list(psi.keys()))
    # Otherwise, take the basis state given as the input
    else:
        sigma = initial_state

    # Record the number of accepted samples
    N_acc = 0

    # A loop will implement the sampling
    for i in range(N):
        sigma_prime = propose_move(sigma)

        # Now accept/reject
        u = np.random.uniform(0, 1)
        A = acc_prob(psi, sigma, sigma_prime)

        if u <= A:
            # The move is accepted with probability = A
            samples.append(sigma_prime)
            sigma = sigma_prime
            N_acc += 1

        else:
            # The move is rejected
            samples.append(sigma)

    # give the burn-ins and the samples as separate lists
    burn_ins, samples = samples[0:n], samples[n:]

    # Fraction of proposed moves that were accepted
    acceptance_rate = N_acc/N

    return burn_ins, samples, acceptance_rate

In [182]:
# This function calculates the probability of obtaining sigma
# from the samples distribution p(sigma)
def probability(samples, sigma):
    samples = np.array(samples)

    return np.sum(samples == sigma) / len(samples)

In [183]:
# Test the sampler

trial_psi = {
    '0011': 0.10,
    '0101': 0.30,
    '0110': -0.20,
    '1001': 0.40,
    '1010': 0.50,
    '1100': -0.60
}
burn_ins, samples, acceptance_rate = metropolis_sampler(
    trial_psi, N=10000, n=1000
)

states = list(trial_psi.keys())
p_tot = 0
for sigma in states:
    print(f'{sigma}, {probability(samples, sigma):.4f}')
    p_tot += probability(samples, sigma)

print(f'{p_tot:.4f}')
print(f"Acceptance rate: {acceptance_rate:.4f}")

0011, 0.0130
0101, 0.1009
0110, 0.0436
1001, 0.1821
1010, 0.2661
1100, 0.3943
1.0000
Acceptance rate: 0.5019


### 3.8 Computing the local energy

The Metropolis sampler gives us configurations distributed according to

$$
p_\theta(\sigma)\propto|\psi_\theta(\sigma)|^2.
$$

For each sampled configuration, we need to calculate its local energy

$$
E_{\mathrm{loc}}(\sigma)
=
\sum_{\sigma'}
H_{\sigma',\sigma}
\frac{\psi_\theta(\sigma')}{\psi_\theta(\sigma)}.
$$

The function `apply_H` developed in the exact diagonalization section already gives us the nonzero elements $H_{\sigma',\sigma}$ for a given basis state. We can therefore use it directly to evaluate the local energy without constructing the full Hamiltonian matrix.

In [184]:
# e_loc is a function that calculates the local energy 
# E_{loc}(\sigma) = \sum_{\sigma'} H_{\sigma', \sigma} \dfrac{\psi(\sigma')}{\psi(\sigma)}.
# Takes the trial wavefunction and a basis state as inputs
# and computes E_{loc}.
def e_loc(trial_psi, basis):
    # Initialize the e_loc value
    e_loc = 0
    n_spins = len(basis)

    # Use apply_H to get the required elements of H.
    lc = apply_H(basis, n_spins)

    # Sum over the nonzero terms generated by H |sigma>.
    for key in lc:
        # Get the coefficients
        key_binary = format(key, f'0{n_spins}b')
        psi_sigma, psi_sigma_prime = bra(trial_psi, basis), bra(trial_psi, key_binary)

        # Compute the contribution to the local energy.
        r = (psi_sigma_prime / psi_sigma) * lc[key]
        e_loc += r

    return e_loc

In [185]:
# Get a quick check on the function
trial_psi = {
    '0011': 0.10,
    '0101': 0.30,
    '0110': -0.20,
    '1001': 0.40,
    '1010': 0.50,
    '1100': -0.60
}
e_loc(trial_psi, '1010')

-1.15

### 3.9 Monte Carlo estimation of the energy

Once the configurations have been sampled and the local energy has been evaluated, the variational energy can be estimated as a simple sample average:

$$
E(\theta)
=
\sum_\sigma p_\theta(\sigma)E_{\mathrm{loc}}(\sigma)
\approx
\frac{1}{M}
\sum_{m=1}^{M}
E_{\mathrm{loc}}(\sigma_m),
$$

where $\{\sigma_m\}$ are configurations obtained from the Metropolis-Hastings chain.

We can therefore estimate the expectation value of the Hamiltonian without explicitly summing over the entire Hilbert space.

In [186]:
# To estimate the expectation value of H in the state |\psi>,
# we use the Monte Carlo estimator
# E = (1/M) \sum_{m} E_{loc}(\sigma_m).

def monte_carlo_energy(trial_psi, n_samples, burn_ins):
    # Given the trial wavefunction, compute the expectation
    # value of energy in this state using n_samples.
    _, samples, _ = metropolis_sampler(trial_psi, n_samples, burn_ins)
    energies = [e_loc(trial_psi, basis) for basis in samples]
    energies = np.array(energies)
    e_expected = np.mean(energies)

    return e_expected

In [187]:
# Let's see what the Monte Carlo method estimates as the
# expectation value of energy for this trial wavefunction
print(f'{monte_carlo_energy(trial_psi, 10000, 1000):.6f}')

-0.350952


### 3.10 Comparing with the exact expectation value

For the small $N=4$ system, we can still evaluate the expectation value directly because the number of basis states in the chosen sector is small.

Using

$$
E
=
\frac{
\sum_\sigma |\psi(\sigma)|^2 E_{\mathrm{loc}}(\sigma)
}{
\sum_\sigma |\psi(\sigma)|^2
},
$$

we can calculate the exact expectation value for the chosen trial wavefunction and compare it with the Monte Carlo estimate.

This provides a useful check that the sampling and local-energy implementations are working correctly.

In [188]:
# Calculate the exact energy for the trial wavefunction
# and compare with the Monte Carlo approxiamtion.

def exact_energy(trial_psi):
    # E_exact = (1 / normalization) \times \sum_{\sigma} |\psi(\sigma)|^2 E_{loc}(\sigma)
    # where normalization = \sum_{\sigma} |\psi(\sigma)|^2
    normalization = 0
    energy = 0
    for key in trial_psi:
        energy += (trial_psi[key] ** 2) * e_loc(trial_psi, key)
        normalization += trial_psi[key] ** 2

    e_exact = energy / normalization

    return e_exact

In [189]:
# Test the exact energy for a small system (N = 4)
trial_psi = {
    '0011': 0.10,
    '0101': 0.30,
    '0110': -0.20,
    '1001': 0.40,
    '1010': 0.50,
    '1100': -0.60
}
print(f'{exact_energy(trial_psi):.6f}')

-0.354396


### 3.11 Statistical convergence of the Monte Carlo estimate

The Monte Carlo estimate is not exactly equal to the expectation value because it is obtained from a finite number of samples.

We therefore expect the result to fluctuate between different runs. Increasing the number of samples should generally reduce the statistical uncertainty, although the samples generated by a Markov chain are correlated and are therefore not equivalent to independent samples.

Because both the number of samples and the number of burn-in steps are varied simultaneously, this experiment is not intended to determine an optimal choice of either parameter. Rather, it illustrates the statistical fluctuations of the Monte Carlo estimate for different sampling conditions.

In [190]:
# Run a few tests and compare the exact values with the
# Monte Carlo approximation.

trial_psi = {
    '0011': 0.10,
    '0101': 0.30,
    '0110': -0.20,
    '1001': 0.40,
    '1010': 0.50,
    '1100': -0.60
}

n_samples_list = [2500, 5000, 10000, 25000, 50000]
burn_ins_list = [100, 250, 500, 750, 1000]

for n_samples in n_samples_list:
    for burn_ins in burn_ins_list:
        exact = exact_energy(trial_psi)
        approx = monte_carlo_energy(trial_psi, n_samples, burn_ins)

        # absolute relative error percentage = |[(monte carlo - exact) / exact]| * 100
        abs_rel_error = abs((approx/exact - 1)) * 100
        print(f'{n_samples:>6} samples and {burn_ins:<4} burn-ins: {abs_rel_error:>7.3f}%')

  2500 samples and 100  burn-ins:   0.135%
  2500 samples and 250  burn-ins:  17.073%
  2500 samples and 500  burn-ins:   0.704%
  2500 samples and 750  burn-ins:   1.133%
  2500 samples and 1000 burn-ins:   9.223%
  5000 samples and 100  burn-ins:   6.155%
  5000 samples and 250  burn-ins:   0.089%
  5000 samples and 500  burn-ins:   3.153%
  5000 samples and 750  burn-ins:   5.433%
  5000 samples and 1000 burn-ins:   0.287%
 10000 samples and 100  burn-ins:   1.528%
 10000 samples and 250  burn-ins:   5.843%
 10000 samples and 500  burn-ins:   6.417%
 10000 samples and 750  burn-ins:   0.886%
 10000 samples and 1000 burn-ins:   2.243%
 25000 samples and 100  burn-ins:   0.882%
 25000 samples and 250  burn-ins:   0.328%
 25000 samples and 500  burn-ins:   1.140%
 25000 samples and 750  burn-ins:   0.921%
 25000 samples and 1000 burn-ins:   2.647%
 50000 samples and 100  burn-ins:   0.114%
 50000 samples and 250  burn-ins:   1.055%
 50000 samples and 500  burn-ins:   1.203%
 50000 samp

### 3.12 Burn-in and equilibration

The Metropolis-Hastings chain is initialized from some arbitrary configuration. In general, this initial configuration is not distributed according to the desired stationary distribution.

The early part of the chain can therefore depend on the initial condition. These initial samples are referred to as burn-in or equilibration samples and are discarded before estimating physical quantities.

The purpose of burn-in is not to make the samples independent. Rather, it is to reduce the influence of the initial condition on the subsequent samples.

For our small $N=4$ system, the target distribution is known exactly, so we can use it to investigate this behavior directly.

We first fix the initial configuration and increase the length of the Markov chain. The empirical distribution can then be compared with the exact distribution.

This experiment illustrates that the empirical probabilities fluctuate for short chains and become more stable as more samples are collected.

In [191]:
# We will compare the values of the empirical probabilities (obtained from the samples)
# with the exact probability computed analytically from trial_psi.
def exact_prob(trial_psi, state):
    # The probability of being at a cetain basis state is given by
    # |\psi(state)|^2 divided by the normalization factor
    norm_factor = [trial_psi[state] ** 2 for state in trial_psi]
    norm_factor = sum(norm_factor)

    return (trial_psi[state] ** 2) / norm_factor

# The next function, will compute the empirical probability of a certain state
# by dividing the number of occurrences of that state by the total number of samples.
def empirical_prob(sample, state):
    sample = np.array(sample)

    return np.mean(sample == state)

In [192]:
# We want to get a better understanding of the origin of burn-in samples
# and how the chain behaves as we start off from a random basis state and
# collect the proposals. We will investigate this phenomena for N = 4 spins.

# Use a trial wavefunction
trial_psi = {
    '0011': 0.10,
    '0101': 0.30,
    '0110': -0.20,
    '1001': 0.40,
    '1010': 0.50,
    '1100': -0.60
}

# Now run a test for a number of sample sizes and compare the probabilities
sample_sizes = [10, 100, 500, 1000, 5000, 10000]

# Iterate and collect data with no burn-ins
print(f"{'Sample size':<14}{'0011':<12}{'0101':<12}{'0110':<12}{'1001':<12}{'1010':<12}{'1100':<12}")

# Take some initial state as basis for analysis
initial = '0101'
for sample_size in sample_sizes:
    _, sample, _ = metropolis_sampler(trial_psi, N = sample_size, n = 0, initial_state = initial)
    row = ""
    for state in trial_psi:
        row += f'{empirical_prob(sample, state):<12f}'
    print(f"{sample_size:<14}{row:<12}")

# The final row is for the exact values
final_row = ""
for state in trial_psi:
    final_row += f'{exact_prob(trial_psi, state):<12f}'
print(f"{'Exact':<14}{final_row}")

# By comparing rows, we can see that our empirical distribution aproaches
# the exact distribution with errors that seem to be decreasing as the 
# size of the sample increases.

Sample size   0011        0101        0110        1001        1010        1100        
10            0.000000    0.200000    0.000000    0.400000    0.100000    0.300000    
100           0.000000    0.090000    0.050000    0.150000    0.250000    0.460000    
500           0.016000    0.122000    0.038000    0.174000    0.230000    0.420000    
1000          0.013000    0.096000    0.048000    0.178000    0.221000    0.444000    
5000          0.010000    0.100800    0.049200    0.172000    0.265000    0.403000    
10000         0.011700    0.102000    0.046300    0.178100    0.267500    0.394400    
Exact         0.010989    0.098901    0.043956    0.175824    0.274725    0.395604    


The empirical probabilities approach the exact probabilities as the number of samples increases, although the convergence is not monotonic because the Monte Carlo estimates fluctuate from run to run.

### 3.13 Investigating the effect of burn-in

To investigate the effect of burn-in separately from the randomness between different Monte Carlo runs, we generate one long Markov chain and analyze the same trajectory after discarding different numbers of initial samples.

For a chosen burn-in $b$, the production samples are

$$
\{\sigma_b,\sigma_{b+1},\ldots,\sigma_{M}\}.
$$

We can then compare the empirical distribution obtained after different values of $b$ with the exact distribution.

For this small system, the effect of changing the burn-in is relatively small compared with the statistical fluctuations of the Monte Carlo estimate. This indicates that the initial transient is short compared with the length of the chain for the particular trial wavefunction and proposal used here.

The experiment also illustrates why burn-in should not simply be chosen as the value that gives the smallest error with respect to the exact answer. Burn-in is intended to reduce dependence on the initial condition, whereas statistical fluctuations remain even after equilibration.

In [193]:
# Now let's investigate the burn-ins further. How do they affect
# the convergence and how many initial samples should be discarded
# as burn-ins to get a better empirical sample?

# Generate the long chain first
burn_ins, samples, acceptance_rate = metropolis_sampler(
    trial_psi,
    N=10000,
    n=0,
    initial_state='0101'
)

# Various values for the number of discarded samples
burn_in_values = [0, 10, 50, 100, 250, 500, 1000]

# Compare the empirical and exact values for each state and each value of burn-in
for burn_in in burn_in_values:
    production = samples[burn_in:]

    print(f"\nBurn-in = {burn_in}")
    for state in trial_psi:
        print(f"{state}: {empirical_prob(production, state):.6f}", f"{exact_prob(trial_psi, state):.6f}")



Burn-in = 0
0011: 0.011600 0.010989
0101: 0.106200 0.098901
0110: 0.045800 0.043956
1001: 0.176200 0.175824
1010: 0.269800 0.274725
1100: 0.390400 0.395604

Burn-in = 10
0011: 0.011612 0.010989
0101: 0.106206 0.098901
0110: 0.045846 0.043956
1001: 0.176276 0.175824
1010: 0.269570 0.274725
1100: 0.390490 0.395604

Burn-in = 50
0011: 0.011658 0.010989
0101: 0.106332 0.098901
0110: 0.046030 0.043956
1001: 0.176080 0.175824
1010: 0.269849 0.274725
1100: 0.390050 0.395604

Burn-in = 100
0011: 0.011616 0.010989
0101: 0.106162 0.098901
0110: 0.045758 0.043956
1001: 0.176566 0.175824
1010: 0.270606 0.274725
1100: 0.389293 0.395604

Burn-in = 250
0011: 0.011590 0.010989
0101: 0.106872 0.098901
0110: 0.046051 0.043956
1001: 0.176205 0.175824
1010: 0.270359 0.274725
1100: 0.388923 0.395604

Burn-in = 500
0011: 0.011579 0.010989
0101: 0.106842 0.098901
0110: 0.045895 0.043956
1001: 0.175895 0.175824
1010: 0.270211 0.274725
1100: 0.389579 0.395604

Burn-in = 1000
0011: 0.011333 0.010989
0101: 0.10

### 3.14 Summary

In this section, we introduced the main ideas behind variational Monte Carlo and implemented the basic sampling machinery needed for later neural quantum states.

Starting from a parameterized trial wavefunction, we rewrote the variational energy as an expectation value over the probability distribution

$$
p_\theta(\sigma)\propto|\psi_\theta(\sigma)|^2.
$$

We then used the Metropolis-Hastings algorithm to sample configurations from this distribution. The Hamiltonian action implemented in the exact diagonalization section was used to calculate the local energy for each sampled configuration, allowing the variational energy to be estimated as a Monte Carlo average.

For the small $N=4$ system, we compared the Monte Carlo results with exact calculations and investigated the effects of sample size and burn-in. These tests provide a basic validation of the VMC machinery.

The main components developed here will be reused when we introduce neural quantum states, where the wavefunction is no longer given explicitly but is instead represented by a parameterized neural network.